# Compound Share Analysis

For each flow (`departure_handover`, `arrival_return`, `arrival_new_stock`), assess  
each compound's share of its regional total — and decide whether shares are stable enough  
to use for disaggregating monthly regional forecasts to compound level.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
cd ..

In [ ]:
events    = pd.read_csv('data/car_events.csv', parse_dates=['event_date'])
compounds = pd.read_csv('data/compounds.csv')

events['month'] = events['event_date'].dt.to_period('M')
events = events.merge(compounds[['compound_id','city','region','total_capacity']], on='compound_id', how='left')

FLOWS = ['departure_handover', 'arrival_return', 'arrival_new_stock']
REGIONS = ['South', 'West', 'North', 'East']

# Monthly counts per compound × flow
monthly = (
    events[events['event_type'].isin(FLOWS)]
    .groupby(['month', 'region', 'compound_id', 'city', 'event_type'])
    .size()
    .reset_index(name='n')
)

# Regional totals per month × flow
region_total = monthly.groupby(['month','region','event_type'])['n'].sum().reset_index(name='region_total')
monthly = monthly.merge(region_total, on=['month','region','event_type'])
monthly['share'] = monthly['n'] / monthly['region_total']


## 1. Overall share stability — CoV per compound × flow

In [ ]:
stability = (
    monthly.groupby(['event_type','region','compound_id'])['share']
    .agg(mean_share='mean', std_share='std', min_share='min', max_share='max', n_months='count')
    .reset_index()
)
stability['cov'] = (stability['std_share'] / stability['mean_share']).round(3)
stability['range'] = (stability['max_share'] - stability['min_share']).round(3)
stability['label'] = pd.cut(
    stability['cov'],
    bins=[-0.001, 0.20, 0.35, 99],
    labels=['stable', 'moderate', 'unstable']
)

SEP = '=' * 70
for flow in FLOWS:
    sub = stability[stability['event_type']==flow].sort_values(['region','mean_share'], ascending=[True,False])
    print(f"\n{SEP}")
    print(f"  {flow}")
    print(f"{SEP}")
    print(sub[['region','compound_id','mean_share','std_share','cov','range','label']]
          .rename(columns={'mean_share':'mean','std_share':'std'})
          .round(3).to_string(index=False))
    counts = sub['label'].value_counts()
    print(f"  → stable: {counts.get('stable',0)}  moderate: {counts.get('moderate',0)}  unstable: {counts.get('unstable',0)}")

## 2. Monthly share time series — visual stability check

In [ ]:
for flow in FLOWS:
    sub = monthly[monthly['event_type'] == flow].copy()
    sub['month_ts'] = sub['month'].dt.to_timestamp()

    regions_in_flow = sorted(sub['region'].unique())
    fig, axes = plt.subplots(1, len(regions_in_flow), figsize=(5*len(regions_in_flow), 4), sharey=False)
    if len(regions_in_flow) == 1:
        axes = [axes]
    fig.suptitle(f'Compound share within region — {flow}', fontsize=12, fontweight='bold')

    colors = plt.cm.tab10.colors
    for ax, region in zip(axes, regions_in_flow):
        rsub = sub[sub['region']==region]
        cpds = sorted(rsub['compound_id'].unique())
        for ci, cpd in enumerate(cpds):
            csub = rsub[rsub['compound_id']==cpd].sort_values('month_ts')
            ax.plot(csub['month_ts'], csub['share'], marker='o', ms=3, lw=1.5,
                    color=colors[ci % 10], label=cpd)
        ax.set_title(f'{region}', fontsize=10)
        ax.set_ylabel('Share of regional total')
        ax.set_ylim(0, 1)
        ax.tick_params(axis='x', rotation=40, labelsize=7)
        ax.legend(fontsize=7, loc='upper right')

    plt.tight_layout()
    fname = f'vis_compound_shares_{flow}.png'
    plt.show()
    